# M3 · Exploración con pandas

**Bootcamp de Datos para Funcionarios Públicos — Formación Pública**
Módulo compartido · Primer módulo del **tronco común** · Se ejecuta en Google Colab.

## ¿Qué vas a lograr?
En el prework de Python básico leíste un archivo CSV y lo recorriste "a mano" usando un bucle `for` fila por fila. Esto funciona para archivos pequeños, pero es lento y requiere escribir mucho código. 

**pandas** es la librería estándar de la Ciencia de Datos en Python. Te permite cargar una planilla de datos completa y analizarla, filtrarla y resumirla utilizando una sola línea de código. En este módulo darás el salto definitivo a manejar **planillas de datos reales** de la administración pública.

**Competencia de salida:** cargar un dataset con pandas, realizar una auditoría visual inicial e inspeccionar filas, columnas, conteos por categoría y métricas estadísticas.

**Entregable:** que las cinco celdas de chequeo de este notebook muestren ✅.


## Los datos que vamos a usar
Trabajaremos con información del sector salud y justicia. Usaremos un dataset real extraído del catálogo nacional de datos abiertos de Chile (datos.gob.cl): los **Servicios Médico Legal (SML)** vigentes en Chile, con su respectiva región, comuna y coordenadas geográficas.

*(Usamos una muestra acotada del dataset del Ministerio de Salud para que este cuaderno sea autocontenido; en la práctica, pandas carga archivos con millones de filas exactamente de la misma manera que aprenderás aquí).* 


## 1. ¿Qué es pandas? La Planilla Excel en Memoria (DataFrame)
**pandas** es una librería que organiza tus datos en un **DataFrame**: una tabla estructurada en filas y columnas.

**La analogía de la planilla activa:**
En el módulo M2, leer un archivo requería abrir y cerrar cajones del archivador para procesar un solo papel a la vez. En pandas, la instrucción `pd.read_csv()` carga la planilla Excel completa de un solo golpe directamente en la memoria activa del programa. 

Por convención estándar, importamos la librería con el apodo de `pd` y guardamos la tabla en una variable llamada `df` (abreviación de *DataFrame*).

*(Nota: Para este módulo, el archivo `establecimientos.csv` ya viene guardado de forma fija y estática en esta carpeta de tu proyecto. Fuente de los datos: Establecimientos de Salud Vigentes del Ministerio de Salud (MINSAL) en datos.gob.cl)*

Ejecuta la celda de abajo para cargar la planilla por primera vez en pandas.

In [ ]:
import os
import urllib.request
import pandas as pd

# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("establecimientos.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/A1-exploracion-con-pandas/establecimientos.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "establecimientos.csv")
        print("establecimientos.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar establecimientos.csv automáticamente. Si estás en Colab, asegúrate de subirlo manualmente.")

# Cargamos el archivo en pandas en una sola línea
df = pd.read_csv("establecimientos.csv")
print(f"Planilla cargada con {len(df)} registros.")
df.head()

## 2. Primer vistazo a la tabla (Auditoría visual)
Antes de empezar a hacer cálculos, es una buena práctica realizar una breve inspección del archivo para saber qué columnas contiene, cuántas filas tiene y si hay algún dato vacío.

Los comandos esenciales son:
*   **`df.head()`**: Dibuja en la pantalla las primeras 5 filas.
*   **`df.shape`**: Te entrega las dimensiones de la planilla en el formato `(filas, columnas)`.
*   **`df.columns`**: Lista los nombres de todas las columnas (los encabezados del Excel).
*   **`df.info()`**: Te da la "hoja de metadatos" del archivo: tipo de datos de cada columna y cantidad de celdas rellenadas (lo que te permite detectar a primera vista si faltan datos en blanco).

In [ ]:
print("Dimensiones (filas, columnas):")
print(df.shape)  # Muestra (38, 5)

print("\nEncabezados de las columnas:")
print(df.columns)

print("\nResumen técnico de la planilla:")
df.info()

## 3. Seleccionar columnas: Ocultar y mostrar en pantalla
En Excel es común ocultar columnas temporalmente para enfocarnos solo en lo que nos interesa. En pandas hacemos esto seleccionando columnas mediante sus nombres:

*   **Una sola columna:** Se escribe entre corchetes simples `df["region"]` (esto devuelve una *Series* de pandas, similar a una lista indexada).
*   **Varias columnas a la vez:** Se pasa una lista de nombres de columnas dentro de corchetes dobles `df[["establecimiento", "comuna"]]` (esto devuelve un sub-DataFrame).

In [ ]:
# Seleccionamos solo la columna de la región y mostramos sus primeros elementos
print(df["region"].head())

print("------------------")

# Creamos una vista de la planilla que solo muestre Establecimiento y Comuna
df[["establecimiento", "comuna"]].head()

## 4. Filtrar filas: El botón de Filtro de Excel
Para quedarnos únicamente con las filas que cumplen una determinada condición (por ejemplo, buscar solo los servicios que están en la región del Maule), usamos la sintaxis de filtrado de pandas.

**La analogía del embudo de Excel:**
Imagina que vas al embudo de la columna `"region"` en Excel y marcas la casilla `"Maule"`. En pandas esto se escribe como:

`df[df["region"] == "Maule"]`

*   `df["region"] == "Maule"` es la condición que evalúa fila por fila (devuelve Verdadero o Falso).
*   El `df[...]` exterior toma esa condición y deja pasar únicamente las filas que dieron Verdadero.

In [ ]:
# Filtramos las filas de la región del Maule
maule = df[df["region"] == "Maule"]

print(f"Cantidad de SML en el Maule: {len(maule)} registros")
maule[["establecimiento", "comuna"]]

## 5. Contar por categoría: La Tabla Dinámica rápida (`value_counts`)
Cuando en Excel necesitas saber cuántos registros hay por cada categoría (por ejemplo, cuántos establecimientos del SML hay en cada región), habitualmente construyes una Tabla Dinámica (Pivot Table).

En pandas existe una instrucción directa para hacer esto en un paso: `df["columna"].value_counts()`. Esto agrupa los datos, los cuenta y los ordena de mayor a menor.

In [ ]:
# Contamos de inmediato la cantidad de SML por cada región en Chile
df["region"].value_counts()

## 6. Estadísticos rápidos: Las sumas del pie de página de Excel
Para columnas que contienen datos numéricos (como en este caso la latitud o longitud), podemos aplicar cálculos rápidos sobre toda la columna sin necesidad de bucles `for`:

*   `df["columna"].mean()`: Promedio de la columna.
*   `df["columna"].max()`: El valor máximo.
*   `df["columna"].min()`: El valor mínimo.

### La auditoría automática: `.describe()`
Si llamas a la instrucción `.describe()` sobre una columna numérica, pandas te entregará una auditoría estadística completa de un solo golpe: promedio, desviación estándar, valor mínimo, percentiles y valor máximo.

In [ ]:
# Analizamos la latitud geográfica de los SML a lo largo del país
print("Latitud promedio:", df["latitud"].mean())
print("Establecimiento más al norte (latitud máxima):", df["latitud"].max())  # Arica
print("Establecimiento más al sur (latitud mínima):", df["latitud"].min())   # Punta Arenas

print("\nReporte descriptivo completo de la columna:")
df["latitud"].describe()

## Errores típicos al empezar (Pistas amigables)
- `KeyError`: Intentaste seleccionar una columna que no existe en el DataFrame. Revisa si hay faltas de ortografía, diferencias en mayúsculas, minúsculas, o si olvidaste colocar tildes (ej. `"region"` vs. `"región"` en los datos reales).
- **Confundir la selección de columna con el filtro de filas:**
  - `df["region"]` te da la columna completa.
  - `df[df["region"] == "Maule"]` te da la tabla filtrada (los corchetes exteriores buscan filas).
- `NameError: name 'pd' is not defined`: Olvidaste ejecutar la celda que contiene la instrucción `import pandas as pd`.

---
# Ejercicios prácticos
Usaremos el DataFrame `df` que cargaste arriba con los establecimientos del SML. Completa cada celda de código donde veas la indicación `# TODO` y ejecuta la celda de chequeo que le sigue para validar tus resultados. ¡Mucho éxito!

## Ejercicio 01 · Inspeccionar la tabla
A partir del DataFrame `df` de los establecimientos, calcula:
- `n_establecimientos` utilizando la instrucción `len()` sobre `df` o accediendo a las dimensiones con `df.shape[0]`.
- `columnas` convirtiendo los nombres de las columnas a una lista mediante `list(df.columns)`.

In [ ]:
n_establecimientos = len(df)
columnas = list(df.columns)

In [ ]:
# Celda de chequeo — Ejercicio 01
import pandas as pd
_df = pd.read_csv("establecimientos.csv")
try:
    assert n_establecimientos == len(_df), "n_establecimientos debe ser el número de filas de df"
    assert list(columnas) == list(_df.columns), "columnas debe ser list(df.columns)"
    print("✅ Correcto. Ejercicio 01 superado.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir una variable:", e)

## Ejercicio 02 · Contar por región
Calcula:
- `por_region` aplicando el conteo por categoría (`value_counts()`) sobre la columna `"region"` de `df`.
- `n_maule` extrayendo del resultado `por_region` el valor correspondiente a la clave `"Maule"`.

In [ ]:
por_region = df["region"].value_counts()
n_maule = por_region["Maule"]

In [ ]:
# Celda de chequeo — Ejercicio 02
try:
    assert por_region is not None, "Aún no calculaste por_region"
    assert por_region["Maule"] == 6, "Debe haber 6 SML en el Maule"
    assert por_region["Valparaíso"] == 4, "Debe haber 4 en Valparaíso"
    assert n_maule == 6, 'n_maule debe ser por_region["Maule"] = 6'
    print("✅ Correcto. Ejercicio 02 superado.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir una variable:", e)
except (KeyError, TypeError) as e:
    print("❌ Revisa por_region (usa value_counts) y el nombre de la región:", e)

## Ejercicio 03 · Estadísticos de una columna
A partir de la columna de la latitud geográfica (`df["latitud"]`), calcula y guarda en las variables correspondientes:
- `lat_promedio` → El promedio de la latitud.
- `lat_norte` → La latitud máxima (más al norte, recuerda que en Chile las latitudes son números negativos, por lo que el valor máximo es el más cercano al ecuador).
- `lat_sur` → La latitud mínima (más al sur).

In [ ]:
lat_promedio = df["latitud"].mean()
lat_norte = df["latitud"].max()
lat_sur = df["latitud"].min()

In [ ]:
# Celda de chequeo — Ejercicio 03
import pandas as pd
_df = pd.read_csv("establecimientos.csv")
try:
    assert lat_promedio is not None, "Aún no calculaste lat_promedio"
    assert round(lat_promedio, 4) == round(_df["latitud"].mean(), 4), "lat_promedio debe ser df['latitud'].mean()"
    assert round(lat_norte, 4) == round(_df["latitud"].max(), 4), "lat_norte debe ser el máximo (.max())"
    assert round(lat_sur, 4) == round(_df["latitud"].min(), 4), "lat_sur debe ser el mínimo (.min())"
    print("✅ Correcto. Ejercicio 03 superado.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir una variable:", e)

## Ejercicio 04 · Filtrar por región
Completa la función `cuantos_en(df, region)` para que **devuelva cuántos establecimientos** hay en la región especificada por el parámetro `region`.

**Consejos:**
- Filtra las filas de `df` donde la columna `"region"` sea igual al parámetro `region`.
- Cuenta cuántos elementos/filas quedaron en la tabla filtrada aplicando `len()` sobre ella.
- Borra la palabra `pass` antes de escribir tu código.

In [ ]:
def cuantos_en(df, region):
    return len(df[df["region"] == region])

In [ ]:
# Celda de chequeo — Ejercicio 04
import pandas as pd
_df = pd.read_csv("establecimientos.csv")
try:
    assert cuantos_en(_df, "Maule") == 6, "El Maule tiene 6"
    assert cuantos_en(_df, "Tarapacá") == 1, "Tarapacá tiene 1"
    assert cuantos_en(_df, "Metropolitana") == 2, "Metropolitana tiene 2"
    assert cuantos_en(_df, "Region Inexistente") == 0, "Una región que no existe debe dar 0"
    print("✅ Correcto. Ejercicio 04 superado. ¡Completaste M3!")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir la función cuantos_en:", e)
except TypeError as e:
    print("❌ Revisa la función (¿filtras y devuelves len?):", e)

## Ejercicio 05 · Interpretar la distribución territorial
Hasta ahora calculaste *cuántos* SML hay por región. Ahora darás el paso clave de cualquier analista de datos: **interpretar qué significa ese número** para una decisión de política pública.

La Región **Metropolitana** es, por lejos, la más poblada de Chile (concentra cerca de un tercio de la población del país). Vas a comparar cuántos SML tiene la Metropolitana frente a la región del **Maule** (mucho menos poblada).

**Primera parte — calcula:**
- `sml_metropolitana` → la cantidad de SML en la región `"Metropolitana"` (usa el conteo por categoría de `df["region"]`).
- `sml_maule` → la cantidad de SML en la región `"Maule"`.

**Segunda parte — interpreta:** mira los dos números que obtuviste y elige la conclusión que mejor se sigue de ellos. Guarda la **letra** (entre comillas) en la variable `conclusion`.

- **A)** La Metropolitana tiene *más* SML que el Maule, lo que confirma que la red de SML se reparte siguiendo el tamaño de la población.
- **B)** A pesar de ser la región más poblada del país, la Metropolitana tiene *menos* SML que el Maule. La cantidad de SML por región **no** sigue al tamaño de la población, sino más bien la cobertura del territorio; esto sugiere revisar si la región más poblada tiene capacidad suficiente.
- **C)** Ambas regiones tienen exactamente la misma cantidad de SML, por lo que la distribución es perfectamente equitativa.

*(Opcional, no se corrige):* en la variable `reflexion` escribe en una frase qué dato adicional pedirías para confirmar tu conclusión (por ejemplo, población por región o número de causas atendidas).

In [ ]:
# Conteo por región y extracción de cada valor
_conteo = df["region"].value_counts()
sml_metropolitana = int(_conteo["Metropolitana"])  # 2
sml_maule = int(_conteo["Maule"])                   # 6

# La Metropolitana, siendo la más poblada, tiene MENOS SML que el Maule:
# la distribución no sigue a la población -> opción B
conclusion = "B"

reflexion = """Pediría la población vigente de cada región (y el número de causas o peritajes atendidos por SML) para confirmar si la Metropolitana está sub-dotada respecto de su demanda real."""

In [ ]:
# Celda de chequeo — Ejercicio 05
import pandas as pd
_df = pd.read_csv("establecimientos.csv")
_vc = _df["region"].value_counts()
_esp_metro = int(_vc["Metropolitana"])
_esp_maule = int(_vc["Maule"])
try:
    assert sml_metropolitana is not None, "Aún no calculaste sml_metropolitana"
    assert sml_maule is not None, "Aún no calculaste sml_maule"
    assert int(sml_metropolitana) == _esp_metro, f"sml_metropolitana debe ser {_esp_metro} (usa value_counts sobre 'region')"
    assert int(sml_maule) == _esp_maule, f"sml_maule debe ser {_esp_maule} (usa value_counts sobre 'region')"
    assert conclusion is not None, "Aún no elegiste la conclusión (asigna \"A\", \"B\" o \"C\")"
    _letra = str(conclusion).strip().upper()
    assert _letra in {"A", "B", "C"}, 'conclusion debe ser una letra: "A", "B" o "C"'
    # La correcta se deduce de los números recomputados: Maule (6) > Metropolitana (2)
    _correcta = "B" if _esp_maule > _esp_metro else ("A" if _esp_metro > _esp_maule else "C")
    assert _letra == _correcta, ("La conclusión no se sigue de los números. Compara sml_metropolitana ("
                                 + str(_esp_metro) + ") con sml_maule (" + str(_esp_maule)
                                 + "): ¿cuál tiene más SML y qué dice eso sobre seguir a la población?")
    print("✅ Correcto. Ejercicio 05 superado. Interpretaste la distribución territorial, no solo la calculaste.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir una variable:", e)
except (KeyError, TypeError) as e:
    print("❌ Revisa el conteo por región y los nombres de las regiones:", e)

---
## ¿Terminaste?
Si las cinco celdas de chequeo muestran ✅ al ejecutarse, **¡felicitaciones por completar el módulo M3!** 🎉

Has realizado con éxito tu primer análisis exploratorio de datos (EDA): cargaste un dataset real, lo inspeccionaste visualmente, contaste frecuencias por categorías, calculaste métricas estadísticas sobre coordenadas espaciales y filtraste filas específicas. Observa cómo pandas te permite hacer lo mismo que en el prework, pero con mucho menos esfuerzo y código.

En el próximo módulo (**M4 · Limpieza de datos**) trabajarás con un dataset "sucio" (que contiene celdas en blanco, duplicaciones de filas y formatos inconsistentes de texto, tal como vienen los datos reales en el sector público) y aprenderás a limpiarlo para dejarlo impecable antes de cualquier análisis.